# Evaluación del impacto institucional


## Objetivo del caso de uso

Este caso de uso tiene como objetivo cuantificar y describir el impacto de la producción académica de una institución a partir de indicadores derivados de fuentes abiertas como OpenAlex y OpenAIRE.

Se busca ofrecer una mirada integradora que permita responder preguntas como:  
- ¿Qué nivel de citación tiene la producción institucional?
- ¿Cuál es la visibilidad (open access) de las publicaciones?
- ¿Qué porcentaje de trabajos se encuentra en el top 10% de su disciplina?
- ¿Cómo ha evolucionado el impacto a lo largo del tiempo?


## Actores y destinatarios

Los principales actores interesados en este análisis son:

- **Gestores institucionales** (secretarías de investigación, posgrados, rectorado), para la toma de decisiones.
- **Investigadores/as**, como insumo para evaluar la proyección de su grupo.
- **Evaluadores externos**, que requieren evidencia del impacto académico.
- **Áreas de ciencia abierta**, que monitorean cumplimiento de políticas de acceso abierto.

Este caso de uso también puede alimentar reportes automáticos o tableros institucionales.

## ¿Qué datos se usan?

Se utilizan datos integrados en el Data Vault desde OpenAlex y OpenAIRE, principalmente:

- Información bibliográfica y de citación de publicaciones institucionales.
- Indicadores normalizados: percentiles, citación, FWCI (Field Weighted Citation Impact).
- Estado de acceso abierto de cada trabajo.

La unidad de análisis son las publicaciones que tengan al menos un autor institucional confirmado.

### Procesamiento de datos

##### Imports y configuración inicial

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

import seaborn as sns
import matplotlib.pyplot as plt



#### Carga de datos


##### OpenAlex

Se cargan los datos:
* Hub de Works
* Hub de DOI
* Links entre Work y DOI
* Satélites de Work

In [ ]:
hub_openalex_work = catalog.load('stg_openalex/hub_openalex_work')
hub_openalex_doi = catalog.load('stg_openalex/hub_openalex_doi')
hub_openalex_pmid = catalog.load('stg_openalex/hub_openalex_pmid')
hub_openalex_pmcid = catalog.load('stg_openalex/hub_openalex_pmcid')
hub_openalex_mag = catalog.load('stg_openalex/hub_openalex_mag')
hub_openalex_author = catalog.load('stg_openalex/hub_openalex_author')
hub_openalex_orcid = catalog.load('stg_openalex/hub_openalex_orcid')

link_openalex_work_doi = catalog.load('stg_openalex/link_openalex_work_doi')
link_openalex_work_pmid = catalog.load('stg_openalex/link_openalex_work_pmid')
link_openalex_work_pmcid = catalog.load('stg_openalex/link_openalex_work_pmcid')
link_openalex_work_mag = catalog.load('stg_openalex/link_openalex_work_mag')
link_openalex_work_author = catalog.load('stg_openalex/link_openalex_work_author')
link_openalex_author_orcid = catalog.load('stg_openalex/link_openalex_author_orcid')

sat_openalex_work = catalog.load('stg_openalex/sat_openalex_work')
sat_openalex_author = catalog.load('stg_openalex/sat_openalex_author')


In [ ]:
sat_openalex_work = sat_openalex_work[['work_hk','type','is_retracted','is_paratext','fwci','cited_by_count','referenced_works_count','publication_year','publication_date','oa_status','oa_url','is_oa','citation_normalized_percentile_is_in_top_10_percent','citation_normalized_percentile_is_in_top_1_percent','citation_normalized_percentile_value','cited_by_percentile_year_max','cited_by_percentile_year_min']]

In [ ]:
sat_openalex_work

In [ ]:
fact_publication_openalex = pd.merge(
    hub_openalex_work,
    sat_openalex_work
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    link_openalex_work_doi,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    hub_openalex_doi,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    link_openalex_work_pmid,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    hub_openalex_pmid,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    link_openalex_work_pmcid,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    hub_openalex_pmcid,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    link_openalex_work_mag,
    how='left'
)

fact_publication_openalex = pd.merge(
    fact_publication_openalex,
    hub_openalex_mag,
    how='left'
)

fact_publication_openalex.drop(columns=['work_doi_hk','work_pmid_hk','work_pmcid_hk','work_mag_hk'], inplace=True)

In [ ]:
fact_publication_openalex

In [ ]:
dim_author_openalex = pd.merge(
    hub_openalex_author,
    sat_openalex_author
).drop(columns=['source','load_datetime','hashdiff'])

dim_author_openalex = pd.merge(
    dim_author_openalex,
    link_openalex_author_orcid,
    how='left'
).drop(columns=['source','load_datetime'])

dim_author_openalex = pd.merge(
    dim_author_openalex,
    hub_openalex_orcid,
    how='left'
).drop(columns=['source','load_datetime','author_orcid_hk'])



In [ ]:
dim_author_openalex

##### IR

Se cargan los datos del:

* Hub de items
* Hub de DOI
* Hub de handles
* links entre items y DOI
* links entre items y handles
* Satelite de items

In [ ]:
hub_ir_item = catalog.load('stg_dspace5/hub_dspace5_item')
hub_ir_doi = catalog.load('stg_dspace5/hub_dspace5_doi')
hub_ir_handle = catalog.load('stg_dspace5/hub_dspace5_handle')

link_ir_item_doi = catalog.load('stg_dspace5/link_dspace5_item_doi')
link_ir_item_handle = catalog.load('stg_dspace5/link_dspace5_item_handle')

sat_ir_item = catalog.load('stg_dspace5/sat_dspace5_item')

##### OpenAire



In [ ]:
hub_openaire_researchproduct = catalog.load('stg_openaire/hub_openaire_researchproduct')
hub_openaire_doi = catalog.load('stg_openaire/hub_openaire_doi')

link_openaire_researchproduct_doi = catalog.load('stg_openaire/link_openaire_researchproduct_doi')
link_openaire_researchproduct_handle = catalog.load('stg_openaire/link_openaire_researchproduct_handle')

sat_openaire_researchproduct = catalog.load('stg_openaire/sat_openaire_researchproduct')

In [ ]:
fact_publication_openaire = pd.merge(
    hub_openaire_researchproduct,
    sat_openaire_researchproduct.drop(columns=['load_datetime','source','hashdiff'])
)

In [ ]:
fact_publication_openaire

#### Integración de datos 

Armo un df con los work de OpenAlex con doi en *df_openalex*.

In [ ]:
df_author_openalex = pd.merge(
    hub_openalex_author,
    sat_openalex_author
)

df_author_openalex[['author_id','display_name','works_count','cited_by_count']]

Armo un df con los work de IR con doi en *df_ir*.

In [ ]:
df_ir = pd.merge(
    hub_ir_item,
    link_ir_item_doi,
)

df_ir = pd.merge(
    df_ir,
    link_ir_item_handle,
)

df_ir = pd.merge(
    df_ir,
    hub_ir_handle,
)

df_ir = pd.merge(
    df_ir,
    link_ir_item_doi,
    how='left'
)

df_ir = pd.merge(
    df_ir,
    hub_ir_doi,
    how='left'
)

df_ir = pd.merge(
    df_ir,
    hub_ir_doi,
    how='left'
)

df_ir = pd.merge(
    df_ir,
    sat_ir_item,
    how='left'
)


In [ ]:
df_ir

Join con los DOI del repositorio

In [ ]:
df_publications = pd.merge(
    fact_publication_openalex,
    df_ir,
    on='doi',
    how='left',
    suffixes=('_openalex', '_dspace')
)

In [ ]:
df_publications

In [ ]:
df_publications[['work_id', 'doi', 'doi_hk_dspace']]

## Casos de uso

### Caso 1: Identificación de publicaciones no integradas al IR

Comparar DOIs de OpenAlex con los del IR para detectar trabajos que podrían integrarse.

##### Procesamiento

In [ ]:
filter_publication = df_publications['doi_hk_dspace'].isna()
df_not_in_ir = df_publications[filter_publication]
df_in_ir = df_publications[~filter_publication]

In [ ]:
df_not_in_ir['is_oa'].value_counts(dropna=False)

In [ ]:
df_not_in_ir['is_oa'].value_counts(normalize=True, dropna=False) * 100

In [ ]:
# Verificación y preparación de datos
counts = df_not_in_ir['is_oa'].value_counts().reindex([True, False])
percentages = (df_not_in_ir['is_oa'].value_counts(normalize=True)
              .reindex([True, False]) * 100)

print("Conteos verificados:")
print(counts)
print("\nPorcentajes verificados:")
print(percentages)

# Configuración del gráfico
plt.figure(figsize=(10, 6))
colors = ['#F44336','#4CAF50']  # Verde para Open Access, Rojo para Restringido

# Gráfico
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index, 
                 palette=colors, width=0.4, order=[True, False],
                 legend=False, dodge=False)

# Personalización avanzada
plt.title('Publicaciones por tipo de acceso\n(Open Access vs Restringido)', 
          fontsize=14, pad=20)
plt.xlabel('Tipo de acceso', labelpad=12, fontsize=12)
plt.ylabel('Número de publicaciones', labelpad=12, fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Sí (Open Access)', 'No (Restringido)'], fontsize=11)

# Altura uniforme para textos (20% sobre el máximo)
text_height = max(counts.values) * 1.20

# Etiquetas
for i, (v, p) in enumerate(zip(counts.values, percentages)):
    ax.text(i, text_height, f"{v:,}\n({p:.1f}%)", 
            ha='center', va='center', fontsize=12,
            bbox=dict(facecolor='white', alpha=0.9, boxstyle='round,pad=0.3'))

# Estilo 
ax.yaxis.grid(True, linestyle=':', linewidth=0.7, alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_alpha(0.4)

plt.ylim(0, max(counts.values)*1.3)
plt.tight_layout()
plt.show()

### Caso 2: Enriquecimiento de publicaciones ya integradas

Para las publicaciones que ya están en el IR, se pueden mejorar sus metadatos con datos de OpenAlex como:

* Agregar métricas de impacto
* Completar datos faltantes
* Detectar errores o inconsistencias entre ambos sistemas

**Agregar métricas de impacto**

cited_by_count, fwci, citation_normalized_percentile_value.

In [ ]:
df_in_ir[['handle','cited_by_count','fwci','citation_normalized_percentile_value']]

Top 10 de artículos más citados (el core de tu pregunta)

In [ ]:
# Top 10 más citados
top_cited = df_in_ir.nlargest(10, 'cited_by_count')[['handle', 'cited_by_count', 'fwci']]

# Gráfico
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=top_cited, y='handle', x='cited_by_count', palette='viridis')

# Personalización
plt.title('Top 10 artículos más citados del repositorio', fontsize=14, pad=20)
plt.xlabel('Número de citas', fontsize=12)
plt.ylabel('Handle del documento', fontsize=12)
ax.bar_label(ax.containers[0], fmt='%d', padding=3)

plt.tight_layout()
plt.show()

 Tabla resumen de métricas clave (complemento ideal)

In [ ]:
# Calculamos métricas agregadas
metrics = df_in_ir.agg({
    'cited_by_count': ['sum', 'mean', 'max'],
    'fwci': ['mean', 'median'],
    'citation_normalized_percentile_value': ['mean']
}).round(2)

print("\nMétricas clave del repositorio:")
print(metrics)

**Detectar errores o inconsistencias entre ambos sistemas**

ej.: título distinto, año de publicación incorrecto, tipo mal asignado

<span style="color:red">TODO - Estan mal procesadas las fechas en landing</span>



In [ ]:
discrepant = df_in_ir[
    pd.to_datetime(df_in_ir['dateissued']).dt.normalize() != 
    pd.to_datetime(df_in_ir['publication_date']).dt.normalize()
]

discrepant[['work_id','publication_date','handle', 'dateissued']]

**Completar datos faltantes**

como publication_date, type, oa_status, language, etc.


### Caso 3: Análisis de acceso abierto en el IR

Comprobar cuántas publicaciones en el IR son realmente OA según OpenAlex (is_oa), y:

Detectar publicaciones cerradas que podrían estar en OA.

Sugerir acciones a tomar con base en oa_status y si hay oa_url

### Caso 4: Detección de publicaciones retracted o paratextos

is_retracted: marcar publicaciones retiradas.

is_paratext: excluir editoriales, prólogos, etc. de métricas.

### Caso 5: Cálculo de indicadores institucionales

Podés agrupar por institución (si tenés el bridge entre publicaciones y autores → y autores con filiación institucional) para calcular:

Impacto promedio (fwci, cited_by_count).

Porcentaje OA.

Porcentaje en top 10% y top 1% mundial (citation_normalized_percentile_is_in_top_*).

